# 01 | Land the release and check every row

*BDCC project, Group G. Payment reconciliation close.*

**Question for this notebook:** can every delivered row be placed in time, and which rows are usable?

1. Locate the release files in a Volume
2. Land the nine CSVs as raw tables (strings, unchanged)
3. Reconcile the delivery manifest
4. Apply the row checks (the classroom checks from P1/P2, adapted to nine sources)
5. Compare with the Docker validator's run record

Nothing is deleted here. A row is labelled, never dropped. Cutoffs, replays and versions come in notebook 03.

**Where this fits in the brief**

- **Configuration A (development extract):** widgets `schema = bdcc_g_dev`, `release = development_G`, `validator_run = dev_initial`.
- **Configuration B (full release):** the same notebook and the same rules, with `schema = bdcc_g_full`, `release = full_G`, `validator_run = full_initial`. Only the widgets change, because the brief asks us to process the full release using the same rules.
- **Classroom pattern:** P1 (land raw as strings, label instead of delete, reconcile the counts) and the P2 handover notebook (compare with the container's run record).

**How to read the comments:** a comment that starts with WHY explains a decision, not the syntax.

## 1. Where are the data and where does the work run?

Each release lives in its own schema, so the development extract and the full release never mix. The widgets at the top select the release:

- Development extract: schema = bdcc_g_dev, release = development_G, validator_run = dev_initial
- Full release: schema = bdcc_g_full, release = full_G, validator_run = full_initial

**Step 1. Run the next cell once.** It creates the schema, a managed Volume named landing and two empty upload folders. It then stops with a message that starts with "Missing in". This is expected, because the files are not uploaded yet.

**Step 2. Upload the files.** Open the Volume in Catalog (your schema, Volumes, landing) and upload:

- the nine CSVs of the release, from BDCC_GROUP_G/development_G/ on your computer into landing/development_G/
- the Docker output (run_record.json and six CSVs), from BDCC_GROUP_G_code/docker/output/dev_initial/ into landing/validator/dev_initial/

For the full release, use the folders full_G and full_initial instead.

**Step 3. Run the cell again.** It now lists the nine files with their sizes, and the rest of the notebook can run.

In [0]:
import json
import re

# WHY widgets: the same notebook must run for the development extract and for the full release
# (and later as a task of the Databricks Job). Nothing below is hard-coded to one release.
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "bdcc_g_dev")
dbutils.widgets.text("release", "development_G")
dbutils.widgets.text("validator_run", "dev_initial")
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
release = dbutils.widgets.get("release").strip()
validator_run = dbutils.widgets.get("validator_run").strip()
# WHY validate the names: these values are pasted into SQL text below. A strict pattern keeps a
# typo from turning into a different statement. Same check as in P1.
for value in (catalog, schema):
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value), "Use a simple catalog and schema identifier"
for value in (release, validator_run):
    assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", value), "Use letters, digits, dash or underscore"

# WHY one schema per release: development and full tables carry the same names. Separate schemas
# make it impossible to mix the 10,000-payment extract with the 2.5-million-payment release.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.landing")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")
# WHY UTC: the close policy attributes a payment to the UTC date of its request. In another
# session time zone, a request shortly before midnight would land on the wrong payment day.
spark.sql("SET TIME ZONE 'UTC'")   # payment days are UTC dates

# Two folders in the Volume: the release exactly as delivered, and the Docker validator's output
# for that release (used in section 5).
release_path = f"/Volumes/{catalog}/{schema}/landing/{release}"
validator_path = f"/Volumes/{catalog}/{schema}/landing/validator/{validator_run}"

# The nine files of the release. The manifest comes first: every later decision about time
# depends on it.
FILES = ["delivery_manifests", "gateway_requests", "processor_events", "refund_events", "settlement_lines",
         "reference_mappings", "terminal_assignments", "merchant_campaigns", "terminal_service_tickets"]
dbutils.fs.mkdirs(release_path)      # the two upload targets exist after the first run
dbutils.fs.mkdirs(validator_path)
# WHY stop when a file is missing: a close built on eight of nine files would look complete and
# be wrong. Fail early, with a message that says what to upload.
listed = {f.name: f.size for f in dbutils.fs.ls(release_path)}
missing = [f"{name}.csv" for name in FILES if f"{name}.csv" not in listed]
if missing:
    raise FileNotFoundError(f"Missing in {release_path}: {missing}")
for name in FILES:
    print(f" - {name}.csv ({listed[name + '.csv']:,} bytes)")
print("Tables are written to", f"{catalog}.{schema}")

## 2. Land the files as raw tables

Every field stays a string, so invalid values stay inspectable and identifiers keep their leading zeros (`000183535`). The CSV in the Volume is not changed. Rerunning overwrites only the `raw_*` tables.

The column list comes from `DATA_DICT.xlsx`. A file whose header differs stops the run.

In [0]:
# WHY a fixed currency list: the dictionary names exactly EUR and GBP. Any other code means the
# unit of the amount is unknown, so the row must not enter a total.
CURRENCIES = ["EUR", "GBP"]   # dictionary: both have 100 minor units per major unit

# WHY a rule table instead of eight hand-written checks: every source gets its checks from one
# definition, so a rule cannot be fixed for one source and forgotten for another. The Docker
# validator holds the same table, and the two implementations must agree on every count.
# One entry per source, taken from DATA_DICT.xlsx. The same rule set runs in the Docker validator.
#   key       : business key scoped to its issuing system (namespace first)
#   version   : version column of a versioned fact
#   effective : time the fact becomes effective
#   required  : identifiers without which a row cannot be attached to a payment
#   stamps / optional_stamps / days / amounts / signed / codes : which column gets which check
#   interval  : start column, end column, type, and the comparison that makes the interval invalid
SOURCES = {
    "gateway_requests": dict(
        fields="""source_row_id delivery_id gateway_system gateway_reference request_timestamp
        customer_id card_token issuer_bank_id terminal_system terminal_id operation_cd
        payment_method_cd service_id channel_cd terminal_country_cd card_country_cd currency_cd
        requested_amount_minor response_cd entry_mode_cd authentication_cd device_id""".split(),
        key=["gateway_system", "gateway_reference"], version=None, effective="request_timestamp",
        required=["gateway_system", "gateway_reference", "terminal_system", "terminal_id"],
        stamps=["request_timestamp"], amounts=["requested_amount_minor"],
        codes={"currency_cd": CURRENCIES}),
    "processor_events": dict(
        fields="""source_row_id delivery_id processor_system event_id version processor_reference
        event_type effective_at amount_minor currency_cd settlement_due_at related_event_id""".split(),
        key=["processor_system", "event_id"], version="version", effective="effective_at",
        required=["processor_system", "event_id", "processor_reference"],
        stamps=["effective_at", "settlement_due_at"], amounts=["amount_minor"],
        codes={"currency_cd": CURRENCIES, "event_type": ["AUTHORISATION", "CAPTURE", "REVERSAL"]}),
    "refund_events": dict(
        fields="""source_row_id delivery_id processor_system refund_id version processor_reference
        capture_event_id refund_status effective_at amount_minor currency_cd""".split(),
        key=["processor_system", "refund_id"], version="version", effective="effective_at",
        required=["processor_system", "refund_id", "processor_reference", "capture_event_id"],
        stamps=["effective_at"], amounts=["amount_minor"],
        codes={"currency_cd": CURRENCIES, "refund_status": ["REQUESTED", "COMPLETED"]}),
    "settlement_lines": dict(
        fields="""source_row_id delivery_id provider_system posting_id version provider_reference
        posting_type effective_at booking_date bank_id currency_cd signed_amount_minor""".split(),
        key=["provider_system", "posting_id"], version="version", effective="effective_at",
        required=["provider_system", "posting_id", "provider_reference"],   # bank_id may be missing
        stamps=["effective_at"], days=["booking_date"], signed=["signed_amount_minor"],
        codes={"currency_cd": CURRENCIES, "posting_type": ["CAPTURE", "REVERSAL", "REFUND"]}),
    "reference_mappings": dict(
        fields="""source_row_id delivery_id mapping_system mapping_id version from_system
        from_reference gateway_system gateway_reference effective_at""".split(),
        key=["mapping_system", "mapping_id"], version="version", effective="effective_at",
        required=["mapping_system", "mapping_id", "from_system", "from_reference",
                  "gateway_system", "gateway_reference"],
        stamps=["effective_at"]),
    "terminal_assignments": dict(
        fields="""source_row_id delivery_id terminal_system assignment_id terminal_id merchant_id
        acquiring_bank_id country_cd valid_from valid_to""".split(),
        key=["terminal_system", "assignment_id"], version=None, effective=None,
        required=["terminal_system", "assignment_id", "terminal_id", "acquiring_bank_id"],
        stamps=["valid_from", "valid_to"], interval=("valid_from", "valid_to", "TIMESTAMP", "<=")),
    "merchant_campaigns": dict(
        fields="""source_row_id delivery_id merchant_system campaign_id merchant_id campaign_start
        campaign_end channel_cd offer_type discount_basis_points""".split(),
        key=["merchant_system", "campaign_id"], version=None, effective=None,
        required=["merchant_system", "campaign_id", "merchant_id"],
        days=["campaign_start", "campaign_end"], amounts=["discount_basis_points"],
        interval=("campaign_start", "campaign_end", "DATE", "<")),
    "terminal_service_tickets": dict(
        fields="""source_row_id delivery_id terminal_system ticket_id terminal_id opened_at
        closed_at issue_category priority ticket_status""".split(),
        key=["terminal_system", "ticket_id"], version=None, effective=None,
        required=["terminal_system", "ticket_id", "terminal_id"],
        stamps=["opened_at"], optional_stamps=["closed_at"],
        interval=("opened_at", "closed_at", "TIMESTAMP", "<")),
}
MANIFEST_FIELDS = "delivery_id source_table source_file delivered_at logical_batch declared_row_count".split()

# WHY read everything as text (inferSchema = False): a typed read would silently turn
# 'not_recorded' into NULL and '000183535' into 183535. As text, bad values stay visible and
# identifiers keep their leading zeros. (P1: the raw fields intentionally remain strings.)
# quote / escape: standard CSV rules, so a quoted field that contains a comma stays one field.
# WHY the header assert: a file with missing or reordered columns must stop the run instead of
# shifting values into the wrong column.
# WHY mode overwrite: a rerun replaces the raw tables. It never appends a second copy.
for name in FILES:
    expected = MANIFEST_FIELDS if name == "delivery_manifests" else SOURCES[name]["fields"]
    raw = (spark.read.option("header", True).option("inferSchema", False)
           .option("quote", '"').option("escape", '"')
           .csv(f"{release_path}/{name}.csv"))
    assert raw.columns == expected, f"{name}.csv: expected the dictionary columns in their supplied order"
    raw.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"raw_{name}")
    print(f"Saved raw_{name}; source CSV is unchanged.")

In [0]:
%sql
-- A first look at one source exactly as delivered. Every value is still text.
SELECT source_row_id, delivery_id, processor_system, event_id, version, processor_reference,
       event_type, effective_at, amount_minor, currency_cd, settlement_due_at, related_event_id
FROM raw_processor_events
ORDER BY source_row_id
LIMIT 8

## 3. Can every row be placed in time?

A source row carries no arrival time of its own. Its `delivery_id` points to the manifest, and the manifest says when that logical delivery became available. Every cutoff decision later depends on this link, so we check it first:

- each manifest row is usable (unique `delivery_id`, known source, real UTC time, integer row count)
- each source row belongs to a manifest delivery of its own source
- each delivery holds exactly its declared number of physical rows (transport repeats included)

In [0]:
# WHY a pattern AND a cast (used below): the pattern alone accepts 2026-99-99T12:00:00Z, the cast
# alone accepts other spellings of a time. Only a value that passes both is a real UTC timestamp
# in the agreed format.
STAMP = "^[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}:[0-9]{2}Z$"
DAY = "^[0-9]{4}-[0-9]{2}-[0-9]{2}$"
known_sources = ", ".join(f"'{name}'" for name in SOURCES)

# Step 1: check the manifest itself. WHY first: the manifest is the only place that says when a
# row became available. Each manifest row gets at most one label (the first failed check wins):
#   missing_identifier    - no delivery_id, so no source row can refer to it
#   duplicate_delivery_id - two rows claim the same id, so its delivery time is ambiguous
#   unknown_source        - names a table or file that is not part of the release
#   invalid_timestamp     - delivered_at is no real time: the delivery cannot be placed at a cutoff
#   invalid_row_count     - declared_row_count is no number: completeness cannot be checked
spark.sql(f'''
CREATE OR REPLACE TABLE stg_delivery_manifests AS
SELECT m.*,
       try_cast(m.delivered_at AS TIMESTAMP) AS delivered_ts,
       try_cast(m.declared_row_count AS BIGINT) AS declared_rows,
       CASE
         WHEN coalesce(trim(m.delivery_id), '') = '' THEN 'missing_identifier'
         WHEN count(*) OVER (PARTITION BY m.delivery_id) > 1 THEN 'duplicate_delivery_id'
         WHEN coalesce(m.source_table, '') NOT IN ({known_sources})
           OR coalesce(m.source_file, '') <> concat(m.source_table, '.csv') THEN 'unknown_source'
         WHEN try_cast(m.delivered_at AS TIMESTAMP) IS NULL
           OR NOT coalesce(m.delivered_at RLIKE '{STAMP}', false) THEN 'invalid_timestamp'
         WHEN NOT coalesce(m.declared_row_count RLIKE '^[0-9]+$', false) THEN 'invalid_row_count'
       END AS manifest_issue
FROM raw_delivery_manifests m
''')

# Step 2: count the physical rows per delivery in every source (one SELECT per source, glued
# together with UNION ALL).
actual = "\nUNION ALL\n".join(
    f"SELECT '{name}' AS source_table, delivery_id, count(*) AS actual_rows FROM raw_{name} GROUP BY delivery_id"
    for name in SOURCES)
# Step 3: compare declared and actual row counts.
# WHY a FULL OUTER JOIN: both directions matter. A manifest delivery with too few rows is
# incomplete (MISMATCH). Rows whose delivery_id is not in the manifest cannot be placed in time
# (UNKNOWN_DELIVERY). An inner join would hide both. Only usable manifest rows take part.
spark.sql(f'''
CREATE OR REPLACE TABLE chk_delivery_counts AS
WITH actual AS (
{actual}
)
SELECT coalesce(m.source_table, a.source_table) AS source_table,
       coalesce(m.delivery_id, a.delivery_id) AS delivery_id,
       m.delivered_at,
       m.declared_rows,
       coalesce(a.actual_rows, 0) AS actual_rows,
       CASE WHEN m.delivery_id IS NULL THEN 'UNKNOWN_DELIVERY'
            WHEN m.declared_rows = coalesce(a.actual_rows, 0) THEN 'MATCH'
            ELSE 'MISMATCH' END AS status
FROM (SELECT * FROM stg_delivery_manifests WHERE manifest_issue IS NULL) m
FULL OUTER JOIN actual a
  ON a.delivery_id = m.delivery_id AND a.source_table = m.source_table
''')

In [0]:
%sql
-- One line per source and status. Expected: only MATCH, and declared_rows = actual_rows.
SELECT source_table, status, count(*) AS deliveries,
       sum(declared_rows) AS declared_rows, sum(actual_rows) AS actual_rows
FROM chk_delivery_counts
GROUP BY source_table, status
ORDER BY source_table, status

In [0]:
# WHY a hard stop: this is the gate. If a row cannot be placed in time or a delivery is
# incomplete, no close for this release can be trusted, so we stop before any amount is summed.
# The Docker validator takes the same decision (gate PASS or HOLD, exit code 1 for HOLD).
bad_manifest = spark.sql("SELECT count(*) AS n FROM stg_delivery_manifests WHERE manifest_issue IS NOT NULL").first()["n"]
bad_deliveries = spark.sql("SELECT count(*) AS n FROM chk_delivery_counts WHERE status <> 'MATCH'").first()["n"]
print("Unusable manifest rows:", bad_manifest, "| deliveries that do not match their declared count:", bad_deliveries)
assert bad_manifest == 0 and bad_deliveries == 0, "HOLD: a row cannot be placed in time or a delivery is incomplete"

**Which payment day and cutoffs does this release target?** The manifest answers it. Routine deliveries arrive daily at 06:00 UTC. The last ones cluster around two instants: exactly 08:00:00 on two consecutive days, plus one second after the later one. Those are C1, C2 and a delivery that must stay out of both closes.

**Assumption to state in the report.** No field in the release names D. We read D = 2026-01-31 from this delivery pattern (C1 = 2026-02-01 08:00 UTC, C2 = 2026-02-02 08:00 UTC). `payment_day` stays a parameter in notebook 03 and in the Docker validator, so a different D needs no code change.

In [0]:
%sql
-- The ten latest delivery times. They show the daily 06:00 routine and the special deliveries
-- at exactly 08:00:00 on two consecutive days (= C1 and C2) plus one at 08:00:01 (after C2).
SELECT delivered_at, count(*) AS deliveries, sum(declared_rows) AS declared_rows,
       count(DISTINCT source_table) AS sources
FROM stg_delivery_manifests
GROUP BY delivered_at
ORDER BY delivered_at DESC
LIMIT 10

## 4. Which records can we use under the checks?

Every row goes through seven checks, adapted from P1 and P2. They run in the order below, and the first one that fails gives the row its label. A row that passes all seven is usable.

1. **missing_identifier.** A key part or a required reference is blank, or a reversal does not name its capture. The row cannot be attached to a payment.
2. **unknown_delivery.** The delivery_id does not belong to a usable manifest delivery of this source. The row cannot be placed before or after a cutoff.
3. **invalid_timestamp.** The value is not a real UTC time. For example, 2026-02-30 and 2026-99-99 fit the pattern and are still no dates. Whether the row was effective by the cutoff cannot be decided.
4. **invalid_amount.** The amount is not a positive whole number of minor units. Provider postings carry a sign and must not be zero. The amount would distort a total.
5. **invalid_version.** The version is not a positive whole number, so the highest version cannot be determined.
6. **invalid_code.** The currency is not EUR or GBP, or an event, status or posting type is not in the data dictionary. The unit or the direction of the money is unknown.
7. **invalid_interval.** An interval ends before it starts. A terminal assignment must also not end at the moment it starts. The effective-dated lookup would return the wrong row.

**A missing value is not always an error.** The data dictionary decides. bank_id may be missing on a provider-only posting, related_event_id is empty unless the event is a reversal, and closed_at is empty for an open ticket.

**Two codes are deliberately not checked:** operation_cd and response_cd. A declined request would be a valid record that falls outside the report population. That is a decision about the population, not a data error.

In [0]:
# Small building blocks. Each returns a piece of SQL that is TRUE when the value is BAD.
# WHY generate the SQL: the same seven checks run on eight sources with different column names.
# One generator = one place to read and one place to fix. The last line of this cell prints one
# generated statement, so the result can be read as plain SQL.
def is_blank(c):
    return f"coalesce(trim(r.{c}), '') = ''"

# Not a real UTC time: fails the cast (2026-02-30, 2026-99-99) or the agreed format.
def not_a_stamp(c):
    return f"(try_cast(r.{c} AS TIMESTAMP) IS NULL OR NOT coalesce(r.{c} RLIKE '{STAMP}', false))"

def not_a_day(c):
    return f"(try_cast(r.{c} AS DATE) IS NULL OR NOT coalesce(r.{c} RLIKE '{DAY}', false))"

# Not a positive whole number: the pattern rejects text, '1.5' and '+5'; <= 0 rejects zero.
def not_positive(c):
    return f"(NOT coalesce(r.{c} RLIKE '^[0-9]+$', false) OR coalesce(try_cast(r.{c} AS BIGINT), 0) <= 0)"

# Provider postings carry their own sign (receipts positive, returns negative). A negative value
# is therefore valid here, zero is not.
def not_signed(c):
    return f"(NOT coalesce(r.{c} RLIKE '^-?[0-9]+$', false) OR coalesce(try_cast(r.{c} AS BIGINT), 0) = 0)"

def any_of(tests):
    return "\n            OR ".join(tests) if tests else "false"

# Builds the statement that creates stg_<source>: the raw row + parsed values + exactly one label.
def staging_sql(name, spec):
    stamps = spec.get("stamps", [])
    optional = spec.get("optional_stamps", [])
    days = spec.get("days", [])
    amounts = spec.get("amounts", [])
    signed = spec.get("signed", [])
    version = spec["version"]

    # Check 1, missing_identifier: a required identifier is blank.
    missing = [is_blank(c) for c in spec["required"]]
    # A reversal must name the capture it reverses. Without it, its amount cannot be subtracted
    # from anything.
    if name == "processor_events":
        missing.append("(r.event_type = 'REVERSAL' AND coalesce(trim(r.related_event_id), '') = '')")
    # Check 3, invalid_timestamp. Optional timestamps are only checked when filled (closed_at of an
    # open ticket is legitimately empty).
    bad_time = ([not_a_stamp(c) for c in stamps]
                + [f"(coalesce(trim(r.{c}), '') <> '' AND {not_a_stamp(c)})" for c in optional]
                + [not_a_day(c) for c in days])
    # Check 4, invalid_amount.
    bad_amount = [not_positive(c) for c in amounts] + [not_signed(c) for c in signed]
    # Check 5, invalid_version: without a readable version the highest version cannot be found.
    bad_version = [not_positive(version)] if version else []
    # Check 6, invalid_code: the value is not in the list the dictionary gives for this column.
    bad_code = []
    for column, allowed in spec.get("codes", {}).items():
        values = ", ".join(f"'{v}'" for v in allowed)
        bad_code.append(f"coalesce(r.{column}, '') NOT IN ({values})")
    # Check 7, invalid_interval: the interval ends before (or, for assignments, when) it starts.
    bad_interval = []
    if spec.get("interval"):
        start, end, kind, operator = spec["interval"]
        bad_interval.append(f"try_cast(r.{end} AS {kind}) {operator} try_cast(r.{start} AS {kind})")

    # Parsed copies (p_<column>) next to the raw text. WHY keep both: the raw text is the evidence,
    # the parsed value is what later notebooks calculate with. try_cast returns NULL, never an error.
    parsed = ([f"try_cast(r.{c} AS TIMESTAMP) AS p_{c}" for c in stamps + optional]
              + [f"try_cast(r.{c} AS DATE) AS p_{c}" for c in days]
              + [f"try_cast(r.{c} AS BIGINT) AS p_{c}" for c in amounts + signed])
    # payload = every business column, that is everything except the two transport columns. Its hash
    # is the fingerprint of the content. Same key + same version + same hash = replay.
    # Same key + same version + different hash = conflict.
    payload = ", ".join(f"coalesce(r.{c}, '')" for c in spec["fields"] if c not in ("source_row_id", "delivery_id"))
    # business_key = issuing system first, then the identifier. An identifier is only unique inside its system.
    key = ", ".join(f"r.{c}" for c in spec["key"])
    # Unversioned sources get version 1, so one snapshot rule works for every source.
    version_no = f"coalesce(try_cast(r.{version} AS BIGINT), 1)" if version else "CAST(1 AS BIGINT)"
    # effective_ts under one name for every source. NULL where a source has no effective time.
    effective = f"try_cast(r.{spec['effective']} AS TIMESTAMP)" if spec["effective"] else "CAST(NULL AS TIMESTAMP)"
    parsed_columns = ",\n       ".join(parsed)
    # The CASE is ordered: the first failed check gives the row its label (as in P1). NULL = usable.
    # WHY a LEFT JOIN to the manifest: a row whose delivery is not in the usable manifest keeps its
    # place and gets the label unknown_delivery (check 2). An inner join would make it disappear.
    return f'''
CREATE OR REPLACE TABLE stg_{name} AS
SELECT r.*,
       m.delivered_at,
       m.delivered_ts,
       {parsed_columns},
       concat_ws('|', {key}) AS business_key,
       {version_no} AS version_no,
       {effective} AS effective_ts,
       sha2(concat_ws('~|~', {payload}), 256) AS payload_hash,
       CASE
         WHEN {any_of(missing)} THEN 'missing_identifier'
         WHEN m.delivery_id IS NULL THEN 'unknown_delivery'
         WHEN {any_of(bad_time)} THEN 'invalid_timestamp'
         WHEN {any_of(bad_amount)} THEN 'invalid_amount'
         WHEN {any_of(bad_version)} THEN 'invalid_version'
         WHEN {any_of(bad_code)} THEN 'invalid_code'
         WHEN {any_of(bad_interval)} THEN 'invalid_interval'
       END AS validation_issue
FROM raw_{name} r
LEFT JOIN stg_delivery_manifests m
  ON m.delivery_id = r.delivery_id AND m.source_table = '{name}' AND m.manifest_issue IS NULL
'''

print(staging_sql("processor_events", SOURCES["processor_events"]))   # read one generated statement

Each stg table (stg_ plus the source name, for example stg_processor_events) keeps the raw text columns unchanged and adds these columns:

- **delivered_at, delivered_ts:** when the row became available. The value comes from the delivery manifest.
- **p_ plus the column name, for example p_effective_at:** the parsed value of a timestamp, date or amount. It is NULL when the text cannot be parsed.
- **business_key:** the key with its issuing system first, for example PR01|E000183535C1.
- **version_no, effective_ts:** the version and the effective time, under the same names for every source.
- **payload_hash:** a fingerprint of all business columns, that is every column except source_row_id and delivery_id. Two rows with the same key, version and fingerprint are the same fact delivered twice.
- **validation_issue:** the name of the first failed check. It is NULL when the row is usable.

In [0]:
# Run the generated statement once per source. CREATE OR REPLACE: a rerun gives the same tables.
for name, spec in SOURCES.items():
    spark.sql(staging_sql(name, spec))
    print(f"Saved stg_{name}")

# One small table with the outcome per source. Section 5 compares it with the Docker validator.
counts = "\nUNION ALL\n".join(
    f"SELECT '{name}' AS source_table, coalesce(validation_issue, 'usable') AS outcome, count(*) AS n_rows "
    f"FROM stg_{name} GROUP BY coalesce(validation_issue, 'usable')"
    for name in SOURCES)
spark.sql(f"CREATE OR REPLACE TABLE chk_row_validation AS\n{counts}")

In [0]:
%sql
-- Rows per source and outcome. 'usable' = passed all seven checks.
SELECT source_table, outcome, n_rows
FROM chk_row_validation
ORDER BY source_table, outcome

In [0]:
%sql
-- The rejected rows themselves, with reason and source reference. Nothing was deleted: they are
-- still rows of stg_processor_events, only labelled.
SELECT validation_issue, source_row_id, delivery_id, business_key, version, event_type,
       effective_at, amount_minor, currency_cd
FROM stg_processor_events
WHERE validation_issue IS NOT NULL
ORDER BY validation_issue, source_row_id
LIMIT 10

## 5. Does the container agree?

The Docker validator applied the same rule set to the same files with different code (plain Python, no Spark). Agreement on every count is the evidence that neither implementation invented or lost a row. The file sizes tie the run record to the files in this Volume.

In [0]:
# Read the validator's run record from the Volume (a small JSON file).
with open(f"{validator_path}/run_record.json", encoding="utf-8") as handle:
    record = json.load(handle)
# WHY these two asserts: the record of an aborted run or of a HOLD run must not serve as reference.
assert record["status"] == "complete", "Check the run record."
assert record["gate"] == "PASS", "The validator put this release on HOLD."
# WHY compare file sizes: it ties the run record to the files in this Volume. If a different
# release had been uploaded, the comparison below would be meaningless.
# Limitation: an equal size is weaker evidence than an equal SHA-256 (the run record stores both).
for file_name, facts in record["inputs"].items():
    assert listed.get(file_name) == facts["bytes"], f"{file_name}: the Volume holds a different file than the validator read"

# Collect this notebook's outcome counts per source.
observed = {}
for row in spark.sql("SELECT source_table, outcome, n_rows FROM chk_row_validation").collect():
    observed.setdefault(row["source_table"], {})[row["outcome"]] = int(row["n_rows"])
# WHY compare reason by reason and not only the totals: two different mistakes can cancel out in
# a total (the same idea as the 100 / 150 example in the brief). Two independent implementations
# (Python in the container, Spark SQL here) agreeing on every count is the evidence that neither
# invented or lost a row.
for name in SOURCES:
    delivered = sum(observed[name].values())
    reasons = {k: v for k, v in observed[name].items() if k != "usable"}
    assert delivered == record["counts"][name]["delivered"], (name, delivered)
    assert reasons == record["rejection_reasons"][name], (name, reasons, record["rejection_reasons"][name])
    print(f"{name}: delivered {delivered:,} | rejected {sum(reasons.values()):,} {reasons}")
print("Row checks reconcile with the validator. Image:", record["image_id"], "| validator", record["validator_version"])

## Key findings to carry forward

- The raw tables are the unchanged delivery. `stg_*` adds labels and parsed values and removes nothing.
- A row is placed in time only through the manifest. A release with an unusable manifest row or an incomplete delivery is on HOLD before any amount is summed.
- "Passes the checks" means usable, not true. A usable row can still be a replay, an old version or half of a conflict. That is notebook 03.

**Next:** `02_profile_sources` (what does one row mean in each source, and which sources does the close need?).